# Direct N-body Simulation with CUDA

This notebook builds a small gravitational N-body simulator in CUDA C++ and uses Python only to inspect and animate the saved positions.

The force calculation has two versions:

1. **Direct global-memory kernel:** one thread computes the acceleration of one body and reads every source body from global memory.
2. **Tiled shared-memory kernel:** each block cooperatively loads source bodies into shared memory and reuses that tile for several target bodies.

Both versions use the same velocity-Verlet integrator. The notebook compares their accelerations with a CPU reference, measures force-kernel time, and animates a disk simulation.

The Python setup cell creates or reuses `nbody_initial_conditions.bin`. The quadtree notebook reads the same file and therefore starts its animation from exactly the same positions, velocities, and masses. The direct and Barnes-Hut trajectories will still differ because Barnes-Hut approximates distant interactions.

Use a Colab runtime with an NVIDIA GPU and CUDA Toolkit 13 or newer. The animation is generated from snapshots written by the CUDA program; file output is outside the force-kernel timing.


## 1. CUDA implementation

The simulation uses normalized units with $G=1$. A massive central body and a light disk begin on approximately circular orbits in the $xy$ plane. The force calculation itself is three-dimensional.

The softening length avoids a singular force at very small separations. It is part of the physical model and is used identically by the CPU reference and both CUDA kernels.

The position/mass records use CUDA 13's `double4_16a` type, which preserves the 16-byte alignment of the former `double4` type.


In [1]:
%%writefile nbody_direct.cu
#include <cuda_runtime.h>

#include <algorithm>
#include <cmath>
#include <chrono>
#include <cstdint>
#include <cstdlib>
#include <fstream>
#include <iostream>
#include <random>
#include <vector>

constexpr int BLOCK_SIZE = 256;
constexpr double GRAVITATIONAL_CONSTANT = 1.0;
constexpr double PI = 3.14159265358979323846;
constexpr unsigned int RANDOM_SEED = 42;

#define CUDA_CHECK(call) do { \
    cudaError_t error = (call); \
    if (error != cudaSuccess) { \
        std::cerr << "CUDA error: " << cudaGetErrorString(error) \
                  << " at " << __FILE__ << ":" << __LINE__ << "\n"; \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)

struct HostState {
    std::vector<double4_16a> position_mass;
    std::vector<double4_16a> velocity;
    std::vector<double4_16a> acceleration;
};

HostState make_disk(int n) {
    HostState state;
    state.position_mass.resize(n);
    state.velocity.resize(n);
    state.acceleration.resize(n);

    const double central_mass = 100.0;
    const double disk_mass = 1.0;
    state.position_mass[0] = make_double4_16a(0.0, 0.0, 0.0, central_mass);
    state.velocity[0] = make_double4_16a(0.0, 0.0, 0.0, 0.0);
    state.acceleration[0] = make_double4_16a(0.0, 0.0, 0.0, 0.0);

    std::mt19937 generator(RANDOM_SEED);
    std::uniform_real_distribution<double> angle_distribution(0.0, 2.0 * PI);
    std::uniform_real_distribution<double> radius_squared_distribution(1.0, 16.0);
    const double particle_mass = disk_mass / (n - 1);

    for (int i = 1; i < n; ++i) {
        const double angle = angle_distribution(generator);
        const double radius = std::sqrt(radius_squared_distribution(generator));
        const double speed = std::sqrt(GRAVITATIONAL_CONSTANT * central_mass / radius);

        const double x = radius * std::cos(angle);
        const double y = radius * std::sin(angle);
        const double vx = -speed * std::sin(angle);
        const double vy = speed * std::cos(angle);

        state.position_mass[i] = make_double4_16a(x, y, 0.0, particle_mass);
        state.velocity[i] = make_double4_16a(vx, vy, 0.0, 0.0);
        state.acceleration[i] = make_double4_16a(0.0, 0.0, 0.0, 0.0);
    }
    return state;
}

HostState load_initial_conditions(const char* filename, int expected_n) {
    std::ifstream input(filename, std::ios::binary);
    std::uint64_t stored_n = 0;
    if (!input.read(reinterpret_cast<char*>(&stored_n), sizeof(stored_n))
        || stored_n != static_cast<std::uint64_t>(expected_n)) {
        std::cerr << "Initial-condition file is missing or has the wrong body count.\n";
        std::exit(EXIT_FAILURE);
    }

    HostState state;
    state.position_mass.resize(expected_n);
    state.velocity.resize(expected_n);
    state.acceleration.resize(expected_n);
    for (int i = 0; i < expected_n; ++i) {
        double values[7];
        if (!input.read(reinterpret_cast<char*>(values), sizeof(values))) {
            std::cerr << "Initial-condition file is incomplete.\n";
            std::exit(EXIT_FAILURE);
        }
        state.position_mass[i] = make_double4_16a(
            values[0], values[1], values[2], values[3]);
        state.velocity[i] = make_double4_16a(
            values[4], values[5], values[6], 0.0);
        state.acceleration[i] = make_double4_16a(0.0, 0.0, 0.0, 0.0);
    }
    return state;
}

double4_16a acceleration_at_host(
    int i, const std::vector<double4_16a>& position_mass, double softening_squared)
{
    const double4_16a target = position_mass[i];
    double ax = 0.0;
    double ay = 0.0;
    double az = 0.0;

    for (int j = 0; j < static_cast<int>(position_mass.size()); ++j) {
        if (i == j) continue;

        const double4_16a source = position_mass[j];
        const double dx = source.x - target.x;
        const double dy = source.y - target.y;
        const double dz = source.z - target.z;
        const double distance_squared =
            dx * dx + dy * dy + dz * dz + softening_squared;
        const double inverse_distance = 1.0 / std::sqrt(distance_squared);
        const double scale =
            GRAVITATIONAL_CONSTANT * source.w *
            inverse_distance * inverse_distance * inverse_distance;

        ax += scale * dx;
        ay += scale * dy;
        az += scale * dz;
    }

    return make_double4_16a(ax, ay, az, 0.0);
}

void initialize_accelerations_cpu(HostState& state, double softening_squared) {
    for (int i = 0; i < static_cast<int>(state.position_mass.size()); ++i) {
        state.acceleration[i] =
            acceleration_at_host(i, state.position_mass, softening_squared);
    }
}

void cpu_velocity_verlet(HostState& state, double dt, double softening_squared) {
    const int n = static_cast<int>(state.position_mass.size());
    std::vector<double4_16a> new_position_mass = state.position_mass;
    std::vector<double4_16a> new_acceleration(n);
    std::vector<double4_16a> new_velocity = state.velocity;

    for (int i = 0; i < n; ++i) {
        new_position_mass[i].x += state.velocity[i].x * dt
            + 0.5 * state.acceleration[i].x * dt * dt;
        new_position_mass[i].y += state.velocity[i].y * dt
            + 0.5 * state.acceleration[i].y * dt * dt;
        new_position_mass[i].z += state.velocity[i].z * dt
            + 0.5 * state.acceleration[i].z * dt * dt;
    }

    for (int i = 0; i < n; ++i) {
        new_acceleration[i] =
            acceleration_at_host(i, new_position_mass, softening_squared);
    }

    for (int i = 0; i < n; ++i) {
        new_velocity[i].x += 0.5 *
            (state.acceleration[i].x + new_acceleration[i].x) * dt;
        new_velocity[i].y += 0.5 *
            (state.acceleration[i].y + new_acceleration[i].y) * dt;
        new_velocity[i].z += 0.5 *
            (state.acceleration[i].z + new_acceleration[i].z) * dt;
    }

    state.position_mass.swap(new_position_mass);
    state.velocity.swap(new_velocity);
    state.acceleration.swap(new_acceleration);
}

__global__ void acceleration_global(
    const double4_16a* __restrict__ position_mass,
    double4_16a* __restrict__ acceleration,
    int n,
    double softening_squared)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;

    const double4_16a target = position_mass[i];
    double ax = 0.0;
    double ay = 0.0;
    double az = 0.0;

    for (int j = 0; j < n; ++j) {
        if (i == j) continue;

        const double4_16a source = position_mass[j];
        const double dx = source.x - target.x;
        const double dy = source.y - target.y;
        const double dz = source.z - target.z;
        const double distance_squared =
            dx * dx + dy * dy + dz * dz + softening_squared;
        const double inverse_distance = 1.0 / sqrt(distance_squared);
        const double scale =
            GRAVITATIONAL_CONSTANT * source.w *
            inverse_distance * inverse_distance * inverse_distance;

        ax += scale * dx;
        ay += scale * dy;
        az += scale * dz;
    }

    acceleration[i] = make_double4_16a(ax, ay, az, 0.0);
}

__global__ void acceleration_tiled(
    const double4_16a* __restrict__ position_mass,
    double4_16a* __restrict__ acceleration,
    int n,
    double softening_squared)
{
    __shared__ double4_16a source_tile[BLOCK_SIZE];

    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    const int lane = threadIdx.x;
    double ax = 0.0;
    double ay = 0.0;
    double az = 0.0;
    double4_16a target = make_double4_16a(0.0, 0.0, 0.0, 0.0);

    if (i < n) target = position_mass[i];

    const int number_of_tiles = (n + BLOCK_SIZE - 1) / BLOCK_SIZE;
    for (int tile = 0; tile < number_of_tiles; ++tile) {
        const int source_index = tile * BLOCK_SIZE + lane;
        if (source_index < n) {
            source_tile[lane] = position_mass[source_index];
        } else {
            source_tile[lane] = make_double4_16a(0.0, 0.0, 0.0, 0.0);
        }
        __syncthreads();

        const int tile_count = min(BLOCK_SIZE, n - tile * BLOCK_SIZE);
        if (i < n) {
            for (int k = 0; k < tile_count; ++k) {
                const int j = tile * BLOCK_SIZE + k;
                if (i == j) continue;

                const double4_16a source = source_tile[k];
                const double dx = source.x - target.x;
                const double dy = source.y - target.y;
                const double dz = source.z - target.z;
                const double distance_squared =
                    dx * dx + dy * dy + dz * dz + softening_squared;
                const double inverse_distance = 1.0 / sqrt(distance_squared);
                const double scale =
                    GRAVITATIONAL_CONSTANT * source.w *
                    inverse_distance * inverse_distance * inverse_distance;

                ax += scale * dx;
                ay += scale * dy;
                az += scale * dz;
            }
        }
        __syncthreads();
    }

    if (i < n) acceleration[i] = make_double4_16a(ax, ay, az, 0.0);
}

__global__ void update_positions(double4_16a* position_mass,
                                 const double4_16a* velocity,
                                 const double4_16a* acceleration,
                                 int n,
                                 double dt)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;

    position_mass[i].x += velocity[i].x * dt
        + 0.5 * acceleration[i].x * dt * dt;
    position_mass[i].y += velocity[i].y * dt
        + 0.5 * acceleration[i].y * dt * dt;
    position_mass[i].z += velocity[i].z * dt
        + 0.5 * acceleration[i].z * dt * dt;
}

__global__ void update_velocities(double4_16a* velocity,
                                  double4_16a* acceleration,
                                  const double4_16a* new_acceleration,
                                  int n,
                                  double dt)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;

    velocity[i].x += 0.5 *
        (acceleration[i].x + new_acceleration[i].x) * dt;
    velocity[i].y += 0.5 *
        (acceleration[i].y + new_acceleration[i].y) * dt;
    velocity[i].z += 0.5 *
        (acceleration[i].z + new_acceleration[i].z) * dt;
    acceleration[i] = new_acceleration[i];
}

enum class ForceMethod { Global, Tiled };

void launch_acceleration(ForceMethod method, const double4_16a* d_position_mass,
                         double4_16a* d_acceleration, int n, double eps2)
{
    const int blocks = (n + BLOCK_SIZE - 1) / BLOCK_SIZE;
    if (method == ForceMethod::Global) {
        acceleration_global<<<blocks, BLOCK_SIZE>>>(
            d_position_mass, d_acceleration, n, eps2);
    } else {
        acceleration_tiled<<<blocks, BLOCK_SIZE>>>(
            d_position_mass, d_acceleration, n, eps2);
    }
    CUDA_CHECK(cudaGetLastError());
}

HostState gpu_one_step(const HostState& input, double dt, double eps2,
                       ForceMethod method)
{
    const int n = static_cast<int>(input.position_mass.size());
    const int blocks = (n + BLOCK_SIZE - 1) / BLOCK_SIZE;
    HostState output = input;

    double4_16a *d_position_mass = nullptr;
    double4_16a *d_velocity = nullptr;
    double4_16a *d_acceleration = nullptr;
    double4_16a *d_new_acceleration = nullptr;

    CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_position_mass), n * sizeof(double4_16a)));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_velocity), n * sizeof(double4_16a)));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_acceleration), n * sizeof(double4_16a)));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_new_acceleration), n * sizeof(double4_16a)));

    CUDA_CHECK(cudaMemcpy(d_position_mass, input.position_mass.data(),
                          n * sizeof(double4_16a), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_velocity, input.velocity.data(),
                          n * sizeof(double4_16a), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_acceleration, input.acceleration.data(),
                          n * sizeof(double4_16a), cudaMemcpyHostToDevice));

    update_positions<<<blocks, BLOCK_SIZE>>>(
        d_position_mass, d_velocity, d_acceleration, n, dt);
    CUDA_CHECK(cudaGetLastError());
    launch_acceleration(method, d_position_mass, d_new_acceleration, n, eps2);
    update_velocities<<<blocks, BLOCK_SIZE>>>(
        d_velocity, d_acceleration, d_new_acceleration, n, dt);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaMemcpy(output.position_mass.data(), d_position_mass,
                          n * sizeof(double4_16a), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(output.velocity.data(), d_velocity,
                          n * sizeof(double4_16a), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(output.acceleration.data(), d_acceleration,
                          n * sizeof(double4_16a), cudaMemcpyDeviceToHost));

    CUDA_CHECK(cudaFree(d_new_acceleration));
    CUDA_CHECK(cudaFree(d_acceleration));
    CUDA_CHECK(cudaFree(d_velocity));
    CUDA_CHECK(cudaFree(d_position_mass));
    return output;
}

double max_state_error(const HostState& a, const HostState& b)
{
    double error = 0.0;
    for (std::size_t i = 0; i < a.position_mass.size(); ++i) {
        error = std::max(error, std::abs(a.position_mass[i].x - b.position_mass[i].x));
        error = std::max(error, std::abs(a.position_mass[i].y - b.position_mass[i].y));
        error = std::max(error, std::abs(a.position_mass[i].z - b.position_mass[i].z));
        error = std::max(error, std::abs(a.velocity[i].x - b.velocity[i].x));
        error = std::max(error, std::abs(a.velocity[i].y - b.velocity[i].y));
        error = std::max(error, std::abs(a.velocity[i].z - b.velocity[i].z));
        error = std::max(error, std::abs(a.acceleration[i].x - b.acceleration[i].x));
        error = std::max(error, std::abs(a.acceleration[i].y - b.acceleration[i].y));
        error = std::max(error, std::abs(a.acceleration[i].z - b.acceleration[i].z));
    }
    return error;
}

double benchmark_force_cpu(int n, int repetitions, double eps2)
{
    HostState input = make_disk(n);
    std::vector<double4_16a> acceleration(n);
    const auto start = std::chrono::steady_clock::now();
    for (int repetition = 0; repetition < repetitions; ++repetition) {
        for (int i = 0; i < n; ++i) {
            acceleration[i] = acceleration_at_host(i, input.position_mass, eps2);
        }
    }
    const auto stop = std::chrono::steady_clock::now();
    double checksum = 0.0;
    for (const double4_16a& value : acceleration) checksum += value.x;
    if (!std::isfinite(checksum)) std::exit(EXIT_FAILURE);
    const double total_ms =
        std::chrono::duration<double, std::milli>(stop - start).count();
    return total_ms / repetitions;
}

double benchmark_force_kernel(ForceMethod method, int n, int repetitions,
                              double eps2)
{
    HostState input = make_disk(n);
    double4_16a *d_position_mass = nullptr;
    double4_16a *d_acceleration = nullptr;

    CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_position_mass), n * sizeof(double4_16a)));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_acceleration), n * sizeof(double4_16a)));
    CUDA_CHECK(cudaMemcpy(d_position_mass, input.position_mass.data(),
                          n * sizeof(double4_16a), cudaMemcpyHostToDevice));

    for (int i = 0; i < 3; ++i) {
        launch_acceleration(method, d_position_mass, d_acceleration, n, eps2);
    }
    CUDA_CHECK(cudaDeviceSynchronize());

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));
    for (int i = 0; i < repetitions; ++i) {
        launch_acceleration(method, d_position_mass, d_acceleration, n, eps2);
    }
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));

    float total_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&total_ms, start, stop));

    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(d_acceleration));
    CUDA_CHECK(cudaFree(d_position_mass));
    return total_ms / repetitions;
}

void write_frame(std::ofstream& output, const std::vector<double4_16a>& position_mass)
{
    for (const double4_16a& p : position_mass) {
        const double xyz[3] = {p.x, p.y, p.z};
        output.write(reinterpret_cast<const char*>(xyz), sizeof(xyz));
    }
}

void run_tiled_trajectory(const HostState& initial, int steps, int save_every,
                          double dt, double eps2, const char* filename)
{
    HostState host = initial;
    const int n = static_cast<int>(host.position_mass.size());
    initialize_accelerations_cpu(host, eps2);

    double4_16a *d_position_mass = nullptr;
    double4_16a *d_velocity = nullptr;
    double4_16a *d_acceleration = nullptr;
    double4_16a *d_new_acceleration = nullptr;
    const int blocks = (n + BLOCK_SIZE - 1) / BLOCK_SIZE;

    CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_position_mass), n * sizeof(double4_16a)));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_velocity), n * sizeof(double4_16a)));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_acceleration), n * sizeof(double4_16a)));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_new_acceleration), n * sizeof(double4_16a)));
    CUDA_CHECK(cudaMemcpy(d_position_mass, host.position_mass.data(),
                          n * sizeof(double4_16a), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_velocity, host.velocity.data(),
                          n * sizeof(double4_16a), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_acceleration, host.acceleration.data(),
                          n * sizeof(double4_16a), cudaMemcpyHostToDevice));

    const std::uint64_t frame_count =
        1 + (steps + save_every - 1) / save_every;
    std::ofstream output(filename, std::ios::binary);
    if (!output) {
        std::cerr << "Could not open trajectory output file.\n";
        std::exit(EXIT_FAILURE);
    }
    const std::uint64_t header[2] =
        {static_cast<std::uint64_t>(n), frame_count};
    output.write(reinterpret_cast<const char*>(header), sizeof(header));

    std::vector<double4_16a> host_positions(n);
    CUDA_CHECK(cudaMemcpy(host_positions.data(), d_position_mass,
                          n * sizeof(double4_16a), cudaMemcpyDeviceToHost));
    write_frame(output, host_positions);

    for (int step = 1; step <= steps; ++step) {
        update_positions<<<blocks, BLOCK_SIZE>>>(
            d_position_mass, d_velocity, d_acceleration, n, dt);
        CUDA_CHECK(cudaGetLastError());
        launch_acceleration(ForceMethod::Tiled, d_position_mass,
                            d_new_acceleration, n, eps2);
        update_velocities<<<blocks, BLOCK_SIZE>>>(
            d_velocity, d_acceleration, d_new_acceleration, n, dt);
        CUDA_CHECK(cudaGetLastError());

        if (step % save_every == 0 || step == steps) {
            CUDA_CHECK(cudaMemcpy(host_positions.data(), d_position_mass,
                                  n * sizeof(double4_16a), cudaMemcpyDeviceToHost));
            write_frame(output, host_positions);
        }
    }

    CUDA_CHECK(cudaDeviceSynchronize());
    output.close();
    CUDA_CHECK(cudaFree(d_new_acceleration));
    CUDA_CHECK(cudaFree(d_acceleration));
    CUDA_CHECK(cudaFree(d_velocity));
    CUDA_CHECK(cudaFree(d_position_mass));
}

int main()
{
    constexpr double dt = 0.001;
    constexpr double softening_squared = 0.02 * 0.02;

    HostState validation_input = make_disk(64);
    initialize_accelerations_cpu(validation_input, softening_squared);
    HostState cpu_reference = validation_input;
    cpu_velocity_verlet(cpu_reference, dt, softening_squared);

    HostState global_result =
        gpu_one_step(validation_input, dt, softening_squared, ForceMethod::Global);
    HostState tiled_result =
        gpu_one_step(validation_input, dt, softening_squared, ForceMethod::Tiled);

    const double global_error = max_state_error(cpu_reference, global_result);
    const double tiled_error = max_state_error(cpu_reference, tiled_result);
    std::cout << "One-step CPU/GPU maximum absolute state error (N=64)\n";
    std::cout << "Global-memory kernel: " << global_error
              << (global_error < 1e-9 ? "  PASS\n" : "  CHECK\n");
    std::cout << "Shared-memory tiled kernel: " << tiled_error
              << (tiled_error < 1e-9 ? "  PASS\n" : "  CHECK\n");

    constexpr int benchmark_n = 2048;
    constexpr int cpu_repetitions = 3;
    constexpr int gpu_repetitions = 10;
    const double cpu_ms = benchmark_force_cpu(
        benchmark_n, cpu_repetitions, softening_squared);
    const double global_ms =
        benchmark_force_kernel(ForceMethod::Global, benchmark_n,
                               gpu_repetitions, softening_squared);
    const double tiled_ms =
        benchmark_force_kernel(ForceMethod::Tiled, benchmark_n,
                               gpu_repetitions, softening_squared);
    std::cout << "\nForce evaluation, N=" << benchmark_n
              << " (CPU average over " << cpu_repetitions
              << ", GPU average over " << gpu_repetitions << ")\n";
    std::cout << "CPU direct: " << cpu_ms << " ms/evaluation\n";
    std::cout << "GPU global-memory: " << global_ms << " ms/evaluation\n";
    std::cout << "GPU shared-memory tiled: " << tiled_ms << " ms/evaluation\n";
    std::cout << "Global/tiled time ratio: " << global_ms / tiled_ms << "\n";

    constexpr int animation_n = 512;
    constexpr int animation_steps = 1200;
    constexpr int save_every = 10;
    const HostState animation_initial = load_initial_conditions(
        "nbody_initial_conditions.bin", animation_n);
    run_tiled_trajectory(animation_initial, animation_steps, save_every,
                         dt, softening_squared, "trajectory.bin");
    std::cout << "\nSaved " << (1 + (animation_steps + save_every - 1) / save_every)
              << " frames for the animation to trajectory.bin\n";
    return 0;
}


Writing nbody_direct.cu


## 2. Create or reuse the shared initial condition

By default, this cell creates a pseudorandom disk with seed 42 and saves it to `nbody_initial_conditions.bin`. Run this cell in both notebooks. If the file already exists, it is reused. To create a different sample, remove the file and change `random_seed`; then run this cell and keep the resulting file for both notebooks.


In [2]:
from pathlib import Path
import numpy as np
import hashlib

body_count = 512
random_seed = 42
initial_file = Path("nbody_initial_conditions.bin")

if not initial_file.exists():
    rng = np.random.default_rng(random_seed)
    central_mass = 100.0
    disk_mass = 1.0
    angle = rng.uniform(0.0, 2.0 * np.pi, body_count - 1)
    radius = np.sqrt(rng.uniform(1.0, 16.0, body_count - 1))
    speed = np.sqrt(central_mass / radius)

    initial = np.zeros((body_count, 7), dtype=np.float64)
    initial[0, 3] = central_mass
    initial[1:, 0] = radius * np.cos(angle)
    initial[1:, 1] = radius * np.sin(angle)
    initial[1:, 3] = disk_mass / (body_count - 1)
    initial[1:, 4] = -speed * np.sin(angle)
    initial[1:, 5] = speed * np.cos(angle)

    with initial_file.open("wb") as file:
        np.asarray([body_count], dtype=np.uint64).tofile(file)
        initial.tofile(file)
    print(f"Created {initial_file} with seed {random_seed}.")
with initial_file.open("rb") as file:
    stored_count = np.fromfile(file, dtype=np.uint64, count=1)
    stored = np.fromfile(file, dtype=np.float64)
if stored_count.size != 1 or stored_count[0] != body_count:
    raise ValueError(f"{initial_file} has the wrong body count.")
if stored.size != body_count * 7:
    raise ValueError(f"{initial_file} has an unexpected size.")
print(f"Using shared initial condition from {initial_file}.")
print("Initial-condition SHA-256:", hashlib.sha256(initial_file.read_bytes()).hexdigest())


Created nbody_initial_conditions.bin with seed 42.
Using shared initial condition from nbody_initial_conditions.bin.
Initial-condition SHA-256: 09e8c2457c7875790a5379afe8b6a8575963839adad5c803bcf83d4c56ec0866


## 3. Compile and run

The program checks one Velocity-Verlet step against a CPU reference, then benchmarks CPU and GPU force evaluations on the same generated sample for the selected particle count. It finally runs the tiled GPU version from the shared initial-condition file and saves position snapshots for the animation.


In [3]:
!nvcc -O3 -std=c++17 -arch=native nbody_direct.cu -o nbody_direct
!./nbody_direct


One-step CPU/GPU maximum absolute state error (N=64)
Global-memory kernel: 2.13163e-14  PASS
Shared-memory tiled kernel: 2.13163e-14  PASS

Force evaluation, N=2048 (CPU average over 3, GPU average over 10)
CPU direct: 26.8861 ms/evaluation
GPU global-memory: 7.6884 ms/evaluation
GPU shared-memory tiled: 6.95316 ms/evaluation
Global/tiled time ratio: 1.10574

Saved 121 frames for the animation to trajectory.bin


## 4. Inspect and save the animation

The binary trajectory contains a small header with the number of bodies and frames, followed by three double-precision coordinates for each body in each frame. The final cell saves a separate GIF named `nbody_animation.gif` in the current working directory and displays only a link to that file. Add the GIF to the repository alongside this notebook if you want the animation to be available from GitHub.


In [4]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

with open("trajectory.bin", "rb") as file:
    n_bodies, n_frames = np.fromfile(file, dtype=np.uint64, count=2)
    trajectory = np.fromfile(file, dtype=np.float64)

trajectory = trajectory.reshape(int(n_frames), int(n_bodies), 3)
print(f"Loaded {n_frames} frames for {n_bodies} bodies.")
print("Position range:", trajectory[:, :, :2].min(), "to", trajectory[:, :, :2].max())


Loaded 121 frames for 512 bodies.
Position range: -3.9730970145996034 to 3.985172524244976


In [5]:
fig, ax = plt.subplots(figsize=(6, 6))
points = ax.scatter(trajectory[0, :, 0], trajectory[0, :, 1], s=5, c="royalblue")
central_body, = ax.plot(
    [trajectory[0, 0, 0]], [trajectory[0, 0, 1]],
    marker="o", linestyle="", markersize=9, color="orange",
    label="Central body"
)
ax.set_xlim(-4.5, 4.5)
ax.set_ylim(-4.5, 4.5)
ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Direct N-body simulation with a tiled CUDA kernel")
ax.legend(loc="upper right")

def update(frame):
    points.set_offsets(trajectory[frame, :, :2])
    central_body.set_data([trajectory[frame, 0, 0]], [trajectory[frame, 0, 1]])
    ax.set_title(f"Direct N-body simulation — frame {frame + 1}/{len(trajectory)}")
    return points, central_body,

animation = FuncAnimation(
    fig, update, frames=len(trajectory), interval=50, blit=True
)

from matplotlib.animation import PillowWriter
from IPython.display import FileLink, display

animation_file = "nbody_animation.gif"
animation.save(animation_file, writer=PillowWriter(fps=20), dpi=100)
plt.close(fig)
print(f"Saved animation to {animation_file}")
display(FileLink(animation_file))


Saved animation to nbody_animation.gif


/content/nbody_animation.gif

## 5. What to observe

The global-memory kernel assigns one thread to each target body and loads every source body during its loop. The tiled kernel loads a block of source positions once into shared memory, then reuses them for the target bodies in the block. Each source tile is separated by a block-wide barrier; all threads reach both barriers, including lanes without a valid target in the last block.

The speed ratio is hardware- and workload-dependent. Tiling reduces repeated global-memory loads, but it also adds synchronization and shared-memory traffic. The animation uses the same tiled force kernel and the same integrator as the CPU reference.
